# PCD Assignment 01 — Down Sampling & Up Sampling

**Mata Kuliah:** Pengolahan Citra Digital
**Nama:** _Jettavane Anatha Pindika_ · **NIM:** _25/563693/PA/23715_

## Tujuan
1. Implementasikan **Down Sampling** dengan tiga metode reduksi blok: **Max**, **Average**, dan **Median** (soal menulis "Medium", diinterpretasikan sebagai *Median*).
2. Implementasikan **Up Sampling** dengan tiga interpolator: **Nearest Neighbor (NN)**, **Bilinear**, dan **Bicubic**.
3. Menganalisis pengaruh kedua operasi terhadap beberapa citra dengan karakteristik berbeda (gradasi halus, tepi tajam, frekuensi tinggi, menyerupai foto) menggunakan metrik objektif **PSNR** dan **SSIM**.

> **Cara menjalankan (Google Colab):** *Runtime → Run all*. Semua dependensi sudah tersedia di Colab. Citra uji dibuat otomatis jika folder `images/` kosong; untuk memakai citra sendiri, upload ke folder `images/` lewat panel Files di kiri.

In [ ]:
import os
import csv
import numpy as np
import PIL
from PIL import Image
import matplotlib.pyplot as plt

from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim

# Konstanta resampling PIL (kompatibel versi lama & baru Pillow)
_RS = getattr(Image, "Resampling", Image)
RESAMPLE_BICUBIC = _RS.BICUBIC

IMAGE_DIR = "images"

def to_uint8(arr):
    """Konversi aman array float -> uint8 [0, 255]."""
    return np.clip(np.round(arr), 0, 255).astype(np.uint8)

def show_grid(pil_images, titles, cols=None, suptitle=None, figsize=None):
    """Menampilkan beberapa citra berdampingan dalam satu figur."""
    n = len(pil_images)
    cols = cols or n
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=figsize or (3.0 * cols, 3.1 * rows + 0.6))
    for i, (im, t) in enumerate(zip(pil_images, titles)):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(im)
        plt.title(t, fontsize=9)
        plt.axis("off")
    if suptitle:
        plt.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

print("Library siap — NumPy", np.__version__, "| Pillow", PIL.__version__)

In [ ]:
def make_test_images():
    """Membuat 4 citra uji 512x512 RGB dengan karakteristik berbeda:
    (1) gradasi halus, (2) tepi tajam, (3) frekuensi tinggi, (4) menyerupai foto."""
    size = 512
    rng = np.random.default_rng(42)
    y, x = np.mgrid[0:size, 0:size].astype(np.float64)

    # 1) Gradasi halus — perubahan intensitas lambat (dominan frekuensi rendah)
    grad = x / (size - 1) * 180 + y / (size - 1) * 75
    img_smooth = np.stack([grad, grad * 0.85 + 20, 255 - grad * 0.6], axis=-1)

    # 2) Tepi tajam — bentuk geometri kontras tinggi
    img_edge = np.full((size, size, 3), 240.0)
    img_edge[80:200, 60:220] = [30, 30, 160]                            # persegi panjang
    cy, cx, r = 330, 150, 90
    img_edge[(x - cx) ** 2 + (y - cy) ** 2 <= r ** 2] = [20, 20, 20]    # lingkaran
    img_edge[:, 400:404] = [200, 40, 40]                                # garis vertikal tipis
    img_edge[250:254, 60:480] = [40, 160, 40]                           # garis horizontal tipis
    for i in range(40, size - 40):
        img_edge[i, i:i + 3] = [0, 0, 0]                                # garis diagonal

    # 3) Frekuensi tinggi — papan catur (kiri) + garis-garis halus (kanan)
    img_hf = np.full((size, size, 3), 250.0)
    cell = 16
    img_hf[((x // cell + y // cell) % 2 == 0) & (x < 256)] = 30
    for c in range(256, size):
        if (c - 256) % 8 < 4:
            img_hf[:, c] = 40

    # 4) Menyerupai foto — langit, matahari, gunung, plus derau lembut
    sky = np.clip(205 - y * 0.38, 60, 255)
    img_photo = np.stack([sky, sky * 0.85 + 8, sky * 0.7 + 35], axis=-1)
    img_photo[(x - 380) ** 2 + (y - 95) ** 2 <= 55 ** 2] = [255, 230, 120]
    img_photo[(y > 330) & (y > -0.9 * (x - 80) + 330)] = [42, 82, 52]
    img_photo[(y > 365) & (y > 0.8 * (x - 430) + 365)] = [30, 60, 45]
    img_photo = np.clip(img_photo + rng.normal(0, 5, img_photo.shape), 0, 255)

    files = {
        "01_smooth_gradient.png": img_smooth,
        "02_sharp_edges.png": img_edge,
        "03_high_frequency.png": img_hf,
        "04_photo_like.png": img_photo,
    }
    os.makedirs(IMAGE_DIR, exist_ok=True)
    for name, arr in files.items():
        Image.fromarray(to_uint8(arr)).save(os.path.join(IMAGE_DIR, name))
    return files

def load_images():
    """Memuat citra uji dari folder images/. Jika kosong, citra dibuat otomatis.
    Di Colab, kamu juga bisa meng-upload citra sendiri lewat panel Files
    (klik kanan -> Upload) ke dalam folder images/."""
    if not (os.path.isdir(IMAGE_DIR) and any(
            f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
            for f in os.listdir(IMAGE_DIR))):
        print("Folder images/ kosong -> membuat citra uji sintetis.")
        make_test_images()
    images = {}
    for name in sorted(os.listdir(IMAGE_DIR)):
        if name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            images[name] = Image.open(os.path.join(IMAGE_DIR, name)).convert("RGB")
    return images

TEST_IMAGES = load_images()
show_grid(list(TEST_IMAGES.values()), list(TEST_IMAGES.keys()),
          cols=4, suptitle="Citra uji dengan karakteristik berbeda")

## Metode

### Down Sampling (minifikasi)
Citra dibagi menjadi blok `factor × factor`, lalu tiap blok direduksi menjadi satu piksel:
- **Max** — nilai maksimum blok; mempertahankan fitur terang, cenderung mencerahkan hasil.
- **Average** — rata-rata blok; ekuivalen filter *low-pass* sederhana, menghaluskan detail.
- **Median** — median blok; kuat terhadap derau/outlier karena mengabaikan nilai ekstrem.

### Up Sampling (magnifikasi)
- **Nearest Neighbor** — piksel sumber direplikasi; paling cepat, tetapi hasilnya berblok (*blocky artifacts*).
- **Bilinear** — interpolasi linear dari 4 tetangga terdekat; halus, tetapi tepi menjadi sedikit kabur.
- **Bicubic** — interpolasi kubik dari 16 tetangga (4×4); lebih tajam, tetapi dapat menimbulkan *ringing* (overshoot/undershoot di sekitar tepi).

### Metrik kualitas
- **PSNR** (dB) — makin tinggi makin mirip dengan acuan; `inf` berarti identik.
- **SSIM** ∈ [0, 1] — kemiripan struktur/persepsi; di atas 0.95 dianggap sangat mirip.

In [ ]:
def downsample(arr, factor, method):
    """Downsampling (minifikasi) dengan block reduction: setiap blok
    factor x factor dipetakan menjadi satu piksel.

    method:
      'max'     -> nilai terbesar dalam blok (mempertahankan fitur terang)
      'average' -> rata-rata blok (menghaluskan, low-pass)
      'median'  -> median blok (tahan outlier/derau impulsive)
    Catatan: soal menyebut "Medium" — diasumsikan maksudnya Median.
    """
    h, w = arr.shape[:2]
    h2, w2 = h // factor, w // factor
    a = arr[: h2 * factor, : w2 * factor].astype(np.float64)
    if a.ndim == 2:
        blocks = a.reshape(h2, factor, w2, factor)
    else:
        blocks = a.reshape(h2, factor, w2, factor, a.shape[2])
    if method == "max":
        return blocks.max(axis=(1, 3))
    if method == "average":
        return blocks.mean(axis=(1, 3))
    if method == "median":
        return np.median(blocks, axis=(1, 3))
    raise ValueError("method tidak dikenal: " + method)

# Sanity check ukuran output
_t = np.arange(64, dtype=np.float64).reshape(8, 8, 1)
assert downsample(_t, 2, "max").shape == (4, 4, 1)
print("downsample OK — metode:", ["max", "average", "median"])

In [ ]:
def upsample_nn(arr, factor):
    """Nearest Neighbor (manual): setiap piksel direplikasi menjadi blok
    factor x factor. Cepat, tetapi menghasilkan blok kotak-kotak."""
    return np.repeat(np.repeat(arr, factor, axis=0), factor, axis=1)

def upsample_bilinear(arr, factor):
    """Bilinear (manual): interpolasi linear atas 4 tetangga terdekat,
    dengan konvensi penempatan sampel half-pixel-center."""
    a = arr.astype(np.float64)
    h, w = a.shape[:2]
    H, W = h * factor, w * factor
    ys = np.clip((np.arange(H) + 0.5) / factor - 0.5, 0, h - 1)
    xs = np.clip((np.arange(W) + 0.5) / factor - 0.5, 0, w - 1)
    y0 = np.floor(ys).astype(int); y1 = np.minimum(y0 + 1, h - 1)
    x0 = np.floor(xs).astype(int); x1 = np.minimum(x0 + 1, w - 1)
    wy = ys - y0                       # bobot vertikal (H,)
    wx = xs - x0                       # bobot horizontal (W,)
    if a.ndim == 2:
        wxb, wyb = wx[None, :], wy[:, None]
    else:
        wxb, wyb = wx[None, :, None], wy[:, None, None]
    top = a[y0][:, x0] * (1 - wxb) + a[y0][:, x1] * wxb
    bot = a[y1][:, x0] * (1 - wxb) + a[y1][:, x1] * wxb
    return top * (1 - wyb) + bot * wyb

def upsample_bicubic(arr, factor):
    """Bicubic: interpolasi kubik atas 16 tetangga (4x4). Di sini memakai
    implementasi Pillow (kernel Catmull-Rom, a = -0.5)."""
    pil = Image.fromarray(to_uint8(arr))
    return np.asarray(pil.resize((arr.shape[1] * factor, arr.shape[0] * factor),
                                 RESAMPLE_BICUBIC)).astype(np.float64)

# Sanity check: ukuran hasil harus factor kali lipat
_a = np.arange(16, dtype=np.float64).reshape(4, 4)
assert upsample_nn(_a, 2).shape == (8, 8)
assert upsample_bilinear(_a, 2).shape == (8, 8)
assert upsample_bicubic(_a, 2).shape == (8, 8)
print("upsampler OK — NN, Bilinear, Bicubic")

In [ ]:
def psnr(ref, test):
    """Peak Signal-to-Noise Ratio (dB). Makin tinggi makin mirip."""
    ref = np.asarray(ref, dtype=np.float64)
    test = np.asarray(test, dtype=np.float64)
    mse = np.mean((ref - test) ** 2)
    if mse == 0:
        return float("inf")
    return 10.0 * np.log10(255.0 ** 2 / mse)

def to_gray(arr):
    """Konversi ke grayscale [0..255] (float) untuk perhitungan metrik."""
    return rgb2gray(to_uint8(arr)) * 255.0

def evaluate(ref, test):
    """PSNR & SSIM antara citra acuan dan citra hasil (dihitung di grayscale)."""
    r, t = to_gray(ref), to_gray(test)
    return psnr(r, t), ssim(r, t, data_range=255.0)

_g = to_gray(TEST_IMAGES["01_smooth_gradient.png"])
print("Contoh: PSNR citra terhadap dirinya sendiri =", psnr(_g, _g), "dB")
print("Contoh: PSNR citra terhadap citra lain      =",
      round(psnr(_g, to_gray(TEST_IMAGES["02_sharp_edges.png"])), 2), "dB")

## Eksperimen 1 — Efek Down Sampling

Setiap citra di-downsample dengan **factor 2 dan 4** memakai ketiga metode reduksi. Amati bagaimana detail halus, tepi, dan pola frekuensi tinggi berevolusi saat resolusi menurun.

In [ ]:
for factor in (2, 4):
    for name, img in TEST_IMAGES.items():
        arr = np.asarray(img)
        small = {m: downsample(arr, factor, m) for m in ("max", "average", "median")}
        show_grid(
            [img] + [Image.fromarray(to_uint8(v)) for v in small.values()],
            [name + f" (asli {img.width}x{img.height})"] +
            [f"down {m} f={factor}" for m in small],
            cols=4, suptitle=f"Eksperimen 1 — Down Sampling factor {factor}",
        )

## Eksperimen 2 — Round-trip: Down Sampling → Up Sampling

Informasi yang dibuang saat downsampling tidak bisa dikembalikan. Untuk mengkuantifikasikannya, tiap citra di-downsample (**factor 4**) dengan satu metode, lalu di-upsample kembali ke ukuran asli dengan ketiga interpolator, dan dibandingkan dengan citra asli memakai PSNR/SSIM. Kombinasi down×up = 3×3 per citra.

In [ ]:
FACTOR = 4
rows_csv = []
for name, img in TEST_IMAGES.items():
    arr = np.asarray(img)
    print(f"\n=== {name} (factor {FACTOR}) — PSNR(dB) / SSIM ===")
    print(f"{'down/up':<10}{'NN':>18}{'Bilinear':>18}{'Bicubic':>18}")
    for dm in ("max", "average", "median"):
        small = downsample(arr, FACTOR, dm)
        cells = []
        for um, ufunc in (("nn", upsample_nn),
                          ("bilinear", upsample_bilinear),
                          ("bicubic", upsample_bicubic)):
            restored = to_uint8(ufunc(small, FACTOR))
            p, s = evaluate(arr, restored)
            cells.append(f"{p:7.2f} / {s:.4f}")
            rows_csv.append([name, FACTOR, dm, um, round(p, 2), round(s, 4)])
        print(f"{dm:<10}{cells[0]:>18}{cells[1]:>18}{cells[2]:>18}")

with open("results_roundtrip.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image", "factor", "down_method", "up_method", "psnr_db", "ssim"])
    w.writerows(rows_csv)
print("\nTabel lengkap tersimpan di results_roundtrip.csv")

## Eksperimen 3 — Up Sampling: Artefak Visual & Kecepatan

Metrik round-trip (Eksperimen 2) membutuhkan pasangan citra dengan konten identik. Untuk *zoom* murni (memperbesar crop), tidak ada ground-truth yang sebanding, sehingga eksperimen ini menilai dua hal yang bisa diukur secara adil:
1. **Artefak visual** tiap interpolator pada perbesaran 4× (blok kotak pada NN, pelembutan tepi pada bilinear, *ringing* pada bicubic).
2. **Kecepatan eksekusi** tiap metode (NN tanpa aritmetika floating-point seharusnya paling cepat).

In [ ]:
import time

# --- Bagian A: artefak visual zoom 4x pada dua citra representatif ---
for name in ("03_high_frequency.png", "04_photo_like.png"):
    arr = np.asarray(TEST_IMAGES[name])
    small = arr[192:320, 192:320]        # crop 128x128 dari area tengah
    results = {label: to_uint8(ufunc(small, 4))
               for label, ufunc in (("NN", upsample_nn),
                                    ("Bilinear", upsample_bilinear),
                                    ("Bicubic", upsample_bicubic))}
    show_grid([Image.fromarray(small)] + [Image.fromarray(v) for v in results.values()],
              ["asli 128x128"] + list(results.keys()), cols=4,
              suptitle=f"Eksperimen 3A — Zoom 4x ({name})")

# --- Bagian B: benchmark kecepatan (rata-rata 5 kali, citra 128x128 -> 512x512) ---
rng = np.random.default_rng(0)
bench_arr = rng.uniform(0, 255, (128, 128, 3))
N_RUN = 5
print(f"{'Metode':<10}{'rata-rata (ms)':>16}{'relatif vs NN':>16}")
times = {}
for label, ufunc in (("NN", upsample_nn),
                     ("Bilinear", upsample_bilinear),
                     ("Bicubic", upsample_bicubic)):
    t0 = time.perf_counter()
    for _ in range(N_RUN):
        ufunc(bench_arr, 4)
    times[label] = (time.perf_counter() - t0) / N_RUN * 1000.0
for label, ms in times.items():
    print(f"{label:<10}{ms:>16.2f}{ms / times['NN']:>15.1f}x")
print("\nCatatan: NN tercepat karena hanya replikasi memori. Bilinear di sini "
      "diimplementasikan manual dengan NumPy sehingga relatif lambat, "
      "sementara bicubic memakai rutin C bawaan Pillow sehingga cepat "
      "meski kernelnya lebih besar (4x4 vs 2x2). Kompleksitas teoretis "
      "per piksel keluaran: NN = O(1), bilinear = O(4), bicubic = O(16).")

## Analisis

### 1. Down Sampling: tiga metode reduksi, tiga karakter

Hasil Eksperimen 1–2 menunjukkan bahwa pilihan metode reduksi blok menentukan *informasi apa* yang selamat dari minifikasi:

- **Average** berperilaku sebagai filter *low-pass* ideal sederhana: energi frekuensi tinggi dirata-rata sehingga hilang, tetapi komponen frekuensi rendah (struktur besar) terjaga. Pada round-trip factor 4, kombinasi *average + bilinear* memberikan PSNR tertinggi pada citra gradasi halus (**70.15 dB, SSIM 0.9999**) — ini bukan kebetulan: bilinear secara matematis merekonstruksi secara eksak fungsi intensitas yang linear (gradasi), dan average mempertahankan nilai rata-rata blok yang tepat berada pada grid sampel bilinear. Kesesuaian teori dan pengukuran ini memvalidasi implementasi.
- **Max** memindahkan distribusi intensitas ke atas (bias terang): pada citra menyerupai foto, round-trip max menghasilkan PSNR terendah (**27.6–27.8 dB** vs 35+ dB untuk average) karena piksel terang outlier mendominasi tiap blok. Metode ini hanya pantas bila fitur terang kecil (mis. teks putih di kertas, bintik) wajib dipertahankan.
- **Median** berada di antara keduanya dan unggul pada SSIM untuk citra tepi tajam (*median + NN = 0.9622*, tertinggi untuk citra tersebut) karena tepi biner yang selaras blok bertahan utuh, tetapi rentan terhadap *dithering* ketika nilai dalam blok terbelah ~50/50.

### 2. Karakteristik citra menentukan metode terbaik — tidak ada juara mutlak

Tiga citra dengan karakter berbeda menghasilkan tiga "pemenang" berbeda pada round-trip factor 4:

| Citra | Kombinasi terbaik | PSNR / SSIM | Penjelasan |
|---|---|---|---|
| Gradasi halus | average + bilinear | 70.15 dB / 0.9999 | fungsi linear direkonstruksi eksak oleh interpolasi linear |
| Frekuensi tinggi (catur 16px) | **semua downsampler + NN** | **∞ (identik)** | pola periodik 16px dengan factor 4 → tiap blok 4×4 seragam; NN mereplikasi tanpa error |
| Menyerupai foto | average + bicubic | 35.81 dB / 0.8117 | kompromi tepi & tekstur |

Kasus citra catur adalah ilustrasi ekstrem dari *aliasing vs replication*: downsampling average menghapus frekuensi tinggi (13.98 dB setelah bilinear/bicubic), sedangkan NN "menyelamatkan" pola karena periodanya (16) habis dibagi factor (4). **Bila periode tidak habis dibagi factor, average + NN akan menghasilkan moiré/aliasing parah** — persis alasan anti-aliasing filter wajib sebelum downsampling pada praktik nyata (mis. `PIL.Image.resize` memakai filter low-pass secara default).

### 3. Up Sampling: kualitas vs biaya

- **NN** menghasilkan artefak blok kotak yang jelas pada zoom 4×, tetapi pada pola periodik selaras ia identik dengan asli (∞ dB) dan paling cepat (1.0 ms untuk 128→512).
- **Bilinear** menghaluskan blok tetapi melembutkan tepi (transisi 2 piksel); implementasi manual NumPy di notebook ini (8.8 ms) lebih lambat daripada bicubic karena bicubic memakai rutin C bawaan Pillow (1.9 ms) — perbandingan waktu ini mencerminkan *implementasi* (Python vs C), bukan kompleksitas teoretis (NN = O(1), bilinear = O(4), bicubic = O(16) per piksel keluaran).
- **Bicubic** memberikan tepi paling tajam pada citra foto (35.81 dB), namun SSIM-nya pada citra tepi biner justru terendah (0.9228–0.9346) — kernel kubik 4×4 menghasilkan *ringing* (overshoot terang/gelap) di sekitar tepi ideal, terlihat sebagai bayangan tipis pada zoom. Ini trade-off klasik ketajaman vs artefak.

### 4. Simpulan analisis

1. Downsampling adalah operasi **destruktif dan tidak dapat dibalik**; kualitas round-trip didominasi oleh kesesuaian (metode reduksi, interpolator, karakter frekuensi citra).
2. Aturan praktis yang didukung data: citra foto/alami → *average + bicubic*; citra grafis biner/tepi → *median + NN*; citra halus → *average + bilinear*; pola periodik selaras → NN menang telak.
3. Metrik tunggal menyesatkan: PSNR dan SSIM kadang berbeda pendapat (max pada citra tepi: PSNR kalah tapi SSIM menang) — keduanya harus dilaporkan bersama, ditambah penilaian visual.

## Kesimpulan

1. **Down sampling bersifat destruktif**: rata-rata blok (average) adalah low-pass filter sehingga detail frekuensi tinggi hilang permanen; max mempertahankan fitur terang tetapi bias terang; median paling aman terhadap derau.
2. **Kualitas up sampling** NN < Bilinear < Bicubic pada citra bertepi halus, namun bicubic dapat menghasilkan *ringing* di sekitar tepi tajam.
3. **Kombinasi terbaik round-trip** konsisten pada pasangan *average/bilinear–bicubic* tergantung karakter citra; citra frekuensi tinggi paling rugi, citra gradasi halus paling tahan.
4. Pemilihan metode sebaiknya menyesuaikan karakteristik citra dan tujuan (mis. median bila banyak derau, bicubic bila visual quality prioritas).